In [1]:
1+1

2

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
## Determenistic approch for guardrails

def determenistic_guard_rail(text:str)->bool:
    """Returns True if content is blocked"""
    banned_words = ["hack","malware","exploit"]
    return any(kw in text.lower() for kw in banned_words)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]
print("===Determenistic guardrail approach===")
for inp in test_inputs:
    blocked = determenistic_guard_rail(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}:{inp}")

===Determenistic guardrail approach===
Blocked:How do I hack into a database?
Allowed:What is the capital of France?
Blocked:Explain how malware spreads


In [4]:
from langchain_groq import ChatGroq

def model_based_guardrail(text:str)->str:
    """ uses a llm for the safety,Returns SAFE OR UNSAFE"""
    model = ChatGroq(model = "llama-3.3-70b-versatile",temperature = 0)
    prompt = f"""is the following input is safe to process? Reply with
    SAFE OR UNSAFE
    Input:{text}"""
    
    result = model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("===Model Based Guardrail===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "UNSAFE"if "UNSAFE" in verdict else "SAFE"
    print(f"{status}:{inp}")

    

===Model Based Guardrail===
UNSAFE:How do I hack into a database?
SAFE:What is the capital of France?
SAFE:Explain how malware spreads


In [20]:
## Langchain has inbuilt middleware PII personally identifiable information

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.tools import tool

@tool
def customer_records(query:str)-> str:
    """Look up customer information"""
    return f"Customer record found for query: {query}"

agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools = [customer_records],
    middleware=[
        PIIMiddleware(
            "email",
            strategy='redact',
            apply_to_input=True
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        ),
    ],
)
print("Agent with PII middleware created sucessfully")

Agent with PII middleware created sucessfully


In [21]:
response = agent.invoke({"messages":[{"role":"user","content":"My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"}]})
response["messages"][-1].content

"I've located your customer record. Is there something specific you'd like to know or change regarding your account?"

In [22]:
response

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='24d3d0e9-e994-4cb1-be50-b062b18b2302'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3979ezzsk', 'function': {'arguments': '{"query":"[REDACTED_EMAIL] and card ****-****-****-5100"}', 'name': 'customer_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 236, 'total_tokens': 266, 'completion_time': 0.069475159, 'completion_tokens_details': None, 'prompt_time': 0.01134514, 'prompt_tokens_details': None, 'queue_time': 0.161270746, 'total_time': 0.080820299}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fda7d-f45b-7b42-ba97-4d45304930b3-0', tool_calls=[{'name': 'customer_records', 'args': {'que

In [25]:
try:
    response = agent.invoke({"messages":[{"role":"user","content":"here is my  api_key is sk-aphgxwvsyhuehhbhsgyzgyudg223"}]})
except Exception as e:
    print(f"Blocked as ecxcepted{e}")
    

In [19]:
response

{'messages': [HumanMessage(content='my api key is sk-aphgxwvsyhuehhbhsgyzgyudg223', additional_kwargs={}, response_metadata={}, id='13bc9d48-15d7-4bb7-a649-8ef26bedfd6e'),
  AIMessage(content="I can't store or use your API key. Is there something else I can help you with?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 228, 'total_tokens': 249, 'completion_time': 0.066548176, 'completion_tokens_details': None, 'prompt_time': 0.016502515, 'prompt_tokens_details': None, 'queue_time': 0.052078143, 'total_time': 0.083050691}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fda7d-74aa-7b53-a10f-92824ee1759f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 228, 'output_tokens': 21, 'total_tokens': 249})]}

In [34]:
## Human in the loop middleware 
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool
@tool
def search_web(query:str)-> str:
    """search the web for information"""
    return f"search results for :{query}"

@tool
def send_mail(to:str,subject:str,body:str)->str:
    """send an email to the receiptent"""
    return f"the emial succesfullt send to :{to}"

@tool
def delete_records(table:str,condition:str)->str:
    """delete records from the database"""
    return f"Deleted records from the table {table} where {condition}"

hit_agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools=[search_web,send_mail,delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_mail":True,
                "delete_records":True,
                "search_web":False
            }
        )
    ],
    checkpointer=InMemorySaver()
)

print("Agent succesfully created with human in the loop middleware")




Agent succesfully created with human in the loop middleware


In [35]:
config = {"configurable":{"thread_id":"1"}}

In [38]:
config = {"configurable":{"thread_id":"1"}}

result = hit_agent.invoke({"messages":[{"role":"user","content":"send an email to the john@email.com about project details"}]},config=config)
print("Agent paused awaiting for human approval")
print(result)

Agent paused awaiting for human approval
{'messages': [HumanMessage(content='send an email to the john@email.com about project details', additional_kwargs={}, response_metadata={}, id='40bfab13-bb37-483a-8042-a412e9e95449'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2bb9f25pg', 'function': {'arguments': '{"body":"Please find the project details in the attached file or below.","subject":"Project Details","to":"john@email.com"}', 'name': 'send_mail'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 347, 'total_tokens': 387, 'completion_time': 0.141942533, 'completion_tokens_details': None, 'prompt_time': 0.044357261, 'prompt_tokens_details': None, 'queue_time': 0.051190158, 'total_time': 0.186299794}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fda97-ff7a-7090-9

In [39]:
approved_result = hit_agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config
)

print("Approved Final Response")
approved_result["messages"][-1].content

Approved Final Response


''

In [40]:
for i, msg in enumerate(approved_result["messages"]):
    print("="*50)
    print(i)
    print(type(msg).__name__)
    print(msg.content)

0
HumanMessage
send an email to the john@email.com about project details
1
AIMessage

2
ToolMessage
the emial succesfullt send to :john@email.com
3
AIMessage

4
HumanMessage
send an email to the john@email.com about project details
5
AIMessage

6
ToolMessage
the emial succesfullt send to :john@email.com
7
AIMessage



In [43]:
config2 = {"configurable":{"thread_id":"2"}}

result = hit_agent.invoke({"messages":[{"role":"user","content":"delete the data from the database where active = false"}]},
                          config=config2)

print("waiting for approval..........")

approved_result = hit_agent.invoke(
    Command(resume={"decisions":[{"type":"reject","reason":"Too risky need a dma review"}]},
            ),
    config=config2
)

print("Rejected approval")
print(approved_result["messages"])

waiting for approval..........
Rejected approval
[HumanMessage(content='delete the data from the database where active = false', additional_kwargs={}, response_metadata={}, id='ac2d21d1-e7bf-4392-8b79-c524cdfdbd77'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1s8z8a034', 'function': {'arguments': '{"condition":"active = false","table":"database"}', 'name': 'delete_records'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 346, 'total_tokens': 369, 'completion_time': 0.06526196, 'completion_tokens_details': None, 'prompt_time': 0.035745332, 'prompt_tokens_details': None, 'queue_time': 0.051306388, 'total_time': 0.101007292}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdaa6-a622-76d2-917b-681ba7e274e8-0', tool_calls=[{'name': 'delete_records', 'args': {'conditio

In [60]:
## Custom Guardrail before agent hook

from typing import Any
from langchain.agents.middleware import AgentState,AgentMiddleware,hook_config
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.runtime import Runtime
from langchain.messages import HumanMessage

class ContentModifiedMiddleware(AgentMiddleware):
    """ Deterministic guardrail: used for blocking the content having banned keywords
    this runs before the process everything ;NO llm cost"""
    def __init__(self,banned_keywords:list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in  banned_keywords]
        
@hook_config(can_jump_to=["end"])
def before_agent(self,state:AgentState,runtime:Runtime) -> dict[str,Any] | None:
    if not state["messages"]:
        return None
    
    first_message = state["messages"][0]
    if not isinstance(first_message, HumanMessage):
        return None
    
    content = first_message.content.lower()
    
    for keyword in self.banned_keywords:
        if keyword in content:
            print(f"Blocked keyword detected{keyword}")
            return{
            "messages":[
                {"role":"assistant",
                 "content":("i cannot process content containing inappropriate content,PLease rephrase your content")}
            ],
                "jump_to":"end"
        }
    return None

@tool
def search_tool(query:str)->str:
    """search in the internet"""
    return f"Results for {query}"

filtered_agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools=[search_tool],
    middleware=[
        ContentModifiedMiddleware(
            banned_keywords=["hack","exploit","malware","jailbreak","bypass"]  
        )
    ]
)
print("Content filter agent created")    

Content filter agent created


In [52]:
result = filtered_agent.invoke({"messages":[{"role":"user","content":"what is machine learning"}]})
result["messages"][-1].content



'Machine learning is a type of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to perform a specific task without using explicit instructions. Instead, the machine learns from the data it is given, and it can improve its performance on the task over time.\n\nThere are several key concepts in machine learning, including:\n\n1. Supervised learning: This type of learning involves training a machine on labeled data, where the correct output is already known. The machine learns to map inputs to outputs based on the labeled data.\n2. Unsupervised learning: This type of learning involves training a machine on unlabeled data, where the machine must find patterns or structure in the data on its own.\n3. Reinforcement learning: This type of learning involves training a machine to take actions in an environment to maximize a reward signal.\n\nSome common applications of machine learning include:\n\n1. Image recognition: Machine learning c

In [61]:
result = filtered_agent.invoke({"messages":[{"role":"user","content":"How to hack a system"}]})
result["messages"][-1].content


"I can't provide information on how to engage in illegal activities such as hacking. Is there anything else I can help you with?"

In [62]:
from langchain_core.messages import HumanMessage

class ContentModifiedMiddleware(AgentMiddleware):

    def __init__(self, banned_keywords):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime):

        if not state["messages"]:
            return None

        last_message = state["messages"][-1]

        if not isinstance(last_message, HumanMessage):
            return None

        content = last_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked keyword detected: {keyword}")

                return {
                    "messages": [
                        {
                            "role": "assistant",
                            "content": "I cannot process content containing inappropriate content. Please rephrase your request."
                        }
                    ],
                    "jump_to": "end",
                }

        return None

In [63]:
filtered_agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools=[search_tool],
    middleware=[
        ContentModifiedMiddleware(
            banned_keywords=["hack","exploit","malware","jailbreak","bypass"]  
        )
    ]
)

In [64]:
result = filtered_agent.invoke({"messages":[{"role":"user","content":"How to hack a system"}]})
result["messages"][-1].content


Blocked keyword detected: hack


'I cannot process content containing inappropriate content. Please rephrase your request.'

In [65]:
result = filtered_agent.invoke({"messages":[{"role":"user","content":"what is machine learning"}]})
result["messages"][-1].content



'Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to perform a specific task without using explicit instructions. Instead, the machine learns from data, identifying patterns and relationships within it, and making decisions or predictions based on that data.\n\nThere are several types of machine learning, including:\n\n1. Supervised learning: The machine is trained on labeled data, where the correct output is already known, and the goal is to learn a mapping between input data and the corresponding output labels.\n2. Unsupervised learning: The machine is trained on unlabeled data, and the goal is to discover patterns, relationships, or groupings within the data.\n3. Reinforcement learning: The machine learns by interacting with an environment and receiving rewards or penalties for its actions, with the goal of maximizing a reward signal.\n\nMachine learning has many applications, including:\n\n1. 